In [3]:
# Core
import numpy as np
import pandas as pd

# ML / preprocessing
from sklearn.model_selection import train_test_split

# Utils
from tqdm import tqdm

In [4]:
# Load dataset
df = pd.read_csv("train.csv")

print("Shape:", df.shape)
df.head()

Shape: (159571, 8)


,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


In [5]:
# Ensure text is string and clean basic issues
df['comment_text'] = df['comment_text'].astype(str).fillna("")

# Optional: strip whitespace
df['comment_text'] = df['comment_text'].str.strip()

# Remove empty rows
df = df[df['comment_text'] != ""]

print("After cleaning:", df.shape)

After cleaning: (159571, 8)


In [11]:
import re

def clean_text(text):
    text = str(text)
    
    # Remove quotes
    text = text.replace('"', '')
    
    # Remove newlines / tabs
    text = re.sub(r'\s+', ' ', text)
    
    # Remove weird wikipedia artifacts (optional but useful)
    text = re.sub(r'\(talk\).*?\)', '', text)
    
    return text.strip()

df['comment_text'] = df['comment_text'].apply(clean_text)

In [24]:
N_CONTEXT = 3  # number of previous messages

contexts = []
comments = []

# Labels
label_cols = ['toxic','severe_toxic','obscene','threat','insult','identity_hate']
labels = []

texts = df['comment_text'].tolist()
y = df[label_cols].values

for i in tqdm(range(len(texts))):
    
    # Skip first few rows (not enough context)
    if i < N_CONTEXT:
        continue
    
    # Get previous N messages
    prev_msgs = texts[i-N_CONTEXT:i]
    
    # Format context (IMPORTANT)
    # context_text = " ".join([f"[CTX] {msg}" for msg in prev_msgs])
    context_text = " ".join([
        f"[CTX{i}] {msg}" for i, msg in enumerate(prev_msgs)
    ])
    
    # Current comment
    current_comment = texts[i]
    
    contexts.append(context_text)
    comments.append(current_comment)
    labels.append(y[i])

# Convert to DataFrame
data = pd.DataFrame({
    "context": contexts,
    "comment": comments
})

# Add labels
for idx, col in enumerate(label_cols):
    data[col] = [lbl[idx] for lbl in labels]

print("New dataset shape:", data.shape)
data.head()

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████| 159571/159571 [00:00<00:00, 638230.44it/s]


New dataset shape: (159568, 8)


,context,comment,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,[CTX0] Explanation Why the edits made under my...,More I can't make any real suggestions on impr...,0,0,0,0,0,0
1,[CTX0] D'aww! He matches this background colou...,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0
2,"[CTX0] Hey man, I'm really not trying to edit ...","Congratulations from me as well, use the tools...",0,0,0,0,0,0
3,[CTX0] More I can't make any real suggestions ...,COCKSUCKER BEFORE YOU PISS AROUND ON MY WORK,1,1,1,0,1,0
4,"[CTX0] You, sir, are my hero. Any chance you r...",Your vandalism to the Matt Shirvington article...,0,0,0,0,0,0


In [7]:
# Inspect a few samples
for i in range(3):
    print("\n======================")
    print("Context:\n", data['context'].iloc[i][:300])
    print("\nComment:\n", data['comment'].iloc[i])
    print("\nLabels:\n", data[label_cols].iloc[i].to_dict())


Context:
 [CTX] Explanation
Why the edits made under my username Hardcore Metallica Fan were reverted? They weren't vandalisms, just closure on some GAs after I voted at New York Dolls FAC. And please don't remove the template from the talk page since I'm retired now.89.205.38.27 [CTX] D'aww! He matches this 

Comment:
 "
More
I can't make any real suggestions on improvement - I wondered if the section statistics should be later on, or a subsection of ""types of accidents""  -I think the references may need tidying so that they are all in the exact same format ie date format etc. I can do that later on, if no-one else does first - if you have any preferences for formatting style on references or want to do it yourself please let me know.

There appears to be a backlog on articles for review so I guess there may be a delay until a reviewer turns up. It's listed in the relevant form eg Wikipedia:Good_article_nominations#Transport  "

Labels:
 {'toxic': 0, 'severe_toxic': 0, 'obscene': 0

In [25]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

MAX_WORDS = 20000
MAX_COMMENT_LEN = 150
MAX_CONTEXT_LEN = 200

# Fit tokenizer on BOTH comment + context
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")

all_text = data['comment'].tolist() + data['context'].tolist()
tokenizer.fit_on_texts(all_text)

print("Vocabulary size:", len(tokenizer.word_index))

Vocabulary size: 209671


In [26]:
# Convert text → sequences
comment_seq = tokenizer.texts_to_sequences(data['comment'])
context_seq = tokenizer.texts_to_sequences(data['context'])

# Pad sequences
X_comment = pad_sequences(comment_seq, maxlen=MAX_COMMENT_LEN, padding='post', truncating='post')
X_context = pad_sequences(context_seq, maxlen=MAX_CONTEXT_LEN, padding='post', truncating='post')

# Labels
y = data[label_cols].values

print("Comment shape:", X_comment.shape)
print("Context shape:", X_context.shape)
print("Labels shape:", y.shape)

Comment shape: (159568, 150)
Context shape: (159568, 200)
Labels shape: (159568, 6)


In [27]:
Xc_train, Xc_val, Xctx_train, Xctx_val, y_train, y_val = train_test_split(
    X_comment,
    X_context,
    y,
    test_size=0.2,
    random_state=42
)

print("Train size:", Xc_train.shape[0])
print("Validation size:", Xc_val.shape[0])

Train size: 127654
Validation size: 31914


In [28]:
print("Toxic ratio:", (y.sum(axis=1) > 0).mean())

Toxic ratio: 0.10168078812794545


In [15]:
# Check if context accidentally includes current comment
for i in range(5):
    if data['comment'].iloc[i] in data['context'].iloc[i]:
        print("⚠️ LEAKAGE FOUND")

In [30]:
from tensorflow.keras.layers import (
    Input, Embedding, GRU, Dense, Dropout,
    Concatenate, Bidirectional
)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.metrics import AUC

In [31]:
# Binary: toxic vs non-toxic (any label)
binary_labels = (y_train.sum(axis=1) > 0).astype(int)

toxic_ratio = binary_labels.mean()

class_weight = {
    0: 1.0,
    1: (1 / toxic_ratio)   # boost toxic class
}

print("Class weights:", class_weight)

Class weights: {0: 1.0, 1: np.float64(9.814253863304375)}


In [46]:
EMBED_DIM = 128

# Shared embedding
embedding = Embedding(input_dim=MAX_WORDS+1, output_dim=EMBED_DIM)

# -------- Comment branch --------
comment_input = Input(shape=(MAX_COMMENT_LEN,), name='comment')
x1 = embedding(comment_input)
x1 = Bidirectional(GRU(128))(x1)
x1 = Dropout(0.4)(x1)

# -------- Context branch --------
context_input = Input(shape=(MAX_CONTEXT_LEN,), name='context')
x2 = embedding(context_input)
x2 = Bidirectional(GRU(128))(x2)
x2 = Dense(256, activation='relu')(x2)
x2 = Dropout(0.4)(x2)

# -------- Merge --------
# merged = Concatenate()([x1, x2])

from tensorflow.keras.layers import Multiply

interaction = Multiply()([x1, x2])
merged = Concatenate()([x1, x2, interaction])

x = Dense(64, activation='relu')(merged)
x = Dropout(0.3)(x)

output = Dense(6, activation='sigmoid')(x)

# Model
model = Model(inputs=[comment_input, context_input], outputs=output)

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=[AUC(name='auc')]
)

model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ comment             │ (None, 150)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ context             │ (None, 200)       │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_5         │ (None, 200, 128)  │  2,560,128 │ comment[0][0],    │
│ (Embedding)         │                   │            │ context[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_11    │ (None, 256)       │    198,144 │ embedding_5[1][0] │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_10    │ (None, 256)       │    198,144 │ embedding_5[0][0] │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, 256)       │     65,792 │ bidirectional_11… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_13          │ (None, 256)       │          0 │ bidirectional_10… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_14          │ (None, 256)       │          0 │ dense_8[0][0]     │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multiply_3          │ (None, 256)       │          0 │ dropout_13[0][0], │
│ (Multiply)          │                   │            │ dropout_14[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_3       │ (None, 768)       │          0 │ dropout_13[0][0], │
│ (Concatenate)       │                   │            │ dropout_14[0][0], │
│                     │                   │            │ multiply_3[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_9 (Dense)     │ (None, 64)        │     49,216 │ concatenate_3[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_15          │ (None, 64)        │          0 │ dense_9[0][0]     │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_10 (Dense)    │ (None, 6)         │        390 │ dropout_15[0][0]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 3,071,814 (11.72 MB)

 Trainable params: 3,071,814 (11.72 MB)

 Non-trainable params: 0 (0.00 B)

In [47]:
callbacks = [
    EarlyStopping(
        monitor='val_loss',
        patience=3,
        restore_best_weights=True
    ),
    ModelCheckpoint(
        'best_model.keras',
        save_best_only=True
    )
]

In [48]:
history = model.fit(
    {
        'comment': Xc_train,
        'context': Xctx_train
    },
    y_train,
    validation_data=(
        {
            'comment': Xc_val,
            'context': Xctx_val
        },
        y_val
    ),
    epochs=15,
    batch_size=256,
    class_weight=class_weight,
    callbacks=callbacks
)

Epoch 1/15
499/499 ━━━━━━━━━━━━━━━━━━━━ 528s 1s/step - auc: 0.9356 - loss: 0.0849 - val_auc: 0.9675 - val_loss: 0.0536
Epoch 2/15
499/499 ━━━━━━━━━━━━━━━━━━━━ 622s 1s/step - auc: 0.9806 - loss: 0.0484 - val_auc: 0.9760 - val_loss: 0.0495
Epoch 3/15
499/499 ━━━━━━━━━━━━━━━━━━━━ 505s 1s/step - auc: 0.9855 - loss: 0.0431 - val_auc: 0.9724 - val_loss: 0.0504
Epoch 4/15
499/499 ━━━━━━━━━━━━━━━━━━━━ 587s 1s/step - auc: 0.9893 - loss: 0.0382 - val_auc: 0.9738 - val_loss: 0.0514
Epoch 5/15
499/499 ━━━━━━━━━━━━━━━━━━━━ 555s 1s/step - auc: 0.9921 - loss: 0.0334 - val_auc: 0.9606 - val_loss: 0.0584


In [49]:
from sklearn.metrics import classification_report

# THRESHOLD = 0.3
THRESHOLDS = np.array([0.3, 0.2, 0.3, 0.05, 0.3, 0.05])

# Predictions
y_pred_probs = model.predict({
    'comment': Xc_val,
    'context': Xctx_val
})

y_pred = (y_pred_probs >= THRESHOLD).astype(int)

# Per-label report
label_cols = ['toxic','severe_toxic','obscene','threat','insult','identity_hate']

for i, label in enumerate(label_cols):
    print(f"\n===== {label} =====")
    print(classification_report(y_val[:, i], y_pred[:, i]))

998/998 ━━━━━━━━━━━━━━━━━━━━ 80s 80ms/step

===== toxic =====
              precision    recall  f1-score   support

           0       0.98      0.98      0.98     28900
           1       0.79      0.79      0.79      3014

    accuracy                           0.96     31914
   macro avg       0.88      0.89      0.88     31914
weighted avg       0.96      0.96      0.96     31914


===== severe_toxic =====
              precision    recall  f1-score   support

           0       1.00      0.99      0.99     31614
           1       0.37      0.73      0.49       300

    accuracy                           0.99     31914
   macro avg       0.68      0.86      0.74     31914
weighted avg       0.99      0.99      0.99     31914


===== obscene =====
              precision    recall  f1-score   support

           0       0.99      0.99      0.99     30223
           1       0.77      0.82      0.79      1691

    accuracy                           0.98     31914
   macro avg       

/Users/midhunmanu/Developer/college/mini_project/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/midhunmanu/Developer/college/mini_project/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/midhunmanu/Developer/college/mini_project/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to contr

In [50]:
def predict_with_context(comment, context):
    seq_c = tokenizer.texts_to_sequences([comment])
    seq_ctx = tokenizer.texts_to_sequences([context])
    
    pad_c = pad_sequences(seq_c, maxlen=MAX_COMMENT_LEN, padding='post')
    pad_ctx = pad_sequences(seq_ctx, maxlen=MAX_CONTEXT_LEN, padding='post')
    
    pred = model.predict({'comment': pad_c, 'context': pad_ctx})[0]
    
    return {label: float(round(p, 3)) for label, p in zip(label_cols, pred)}


comment = "you're dead to me"

print("\nNo context:")
print(predict_with_context(comment, ""))

print("\nFriendly context:")
print(predict_with_context(comment, "[CTX] haha bro you're amazing 😂"))

print("\nHostile context:")
print(predict_with_context(comment, "[CTX] I will hurt you [CTX] watch yourself"))


No context:
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
{'toxic': 0.3109999895095825, 'severe_toxic': 0.04899999871850014, 'obscene': 0.13899999856948853, 'threat': 0.04899999871850014, 'insult': 0.15299999713897705, 'identity_hate': 0.07199999690055847}

Friendly context:
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
{'toxic': 0.25999999046325684, 'severe_toxic': 0.008999999612569809, 'obscene': 0.061000000685453415, 'threat': 0.014999999664723873, 'insult': 0.07800000160932541, 'identity_hate': 0.024000000208616257}

Hostile context:
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
{'toxic': 0.27300000190734863, 'severe_toxic': 0.014999999664723873, 'obscene': 0.07699999958276749, 'threat': 0.020999999716877937, 'insult': 0.09300000220537186, 'identity_hate': 0.03200000151991844}


In [51]:
import sqlite3

conn = sqlite3.connect("toxic_flagger.db")
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS messages (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    thread_id TEXT,
    message_text TEXT,
    
    toxic REAL,
    severe_toxic REAL,
    obscene REAL,
    threat REAL,
    insult REAL,
    identity_hate REAL,
    
    user_flagged INTEGER DEFAULT NULL
)
""")

conn.commit()

In [52]:
def save_message(conn, thread_id, text, scores):
    cursor = conn.cursor()
    
    cursor.execute("""
    INSERT INTO messages (
        thread_id, message_text,
        toxic, severe_toxic, obscene,
        threat, insult, identity_hate
    )
    VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        thread_id, text,
        scores['toxic'], scores['severe_toxic'],
        scores['obscene'], scores['threat'],
        scores['insult'], scores['identity_hate']
    ))
    
    conn.commit()
    
    return cursor.lastrowid

In [53]:
def update_feedback(conn, message_id, user_flagged):
    cursor = conn.cursor()
    
    cursor.execute("""
    UPDATE messages
    SET user_flagged = ?
    WHERE id = ?
    """, (user_flagged, message_id))
    
    conn.commit()

In [59]:
DEFAULT_THRESHOLD = 0.3

def get_user_threshold(conn):
    cursor = conn.cursor()
    
    cursor.execute("""
    SELECT user_flagged, toxic
    FROM messages
    WHERE user_flagged IS NOT NULL
    ORDER BY id DESC
    LIMIT 50
    """)
    
    rows = cursor.fetchall()
    
    if len(rows) < 1:
        return DEFAULT_THRESHOLD
    
    disagreements = 0
    
    for user_flagged, toxic_score in rows:
        model_flagged = toxic_score >= DEFAULT_THRESHOLD
        
        if model_flagged != user_flagged:
            disagreements += 1
    
    error_rate = disagreements / len(rows)
    
    if error_rate > 0.3:
        return 0.2   # more sensitive
    elif error_rate < 0.1:
        return 0.4   # more tolerant
    
    return DEFAULT_THRESHOLD

In [55]:
def process_message(conn, thread_id, text, context):
    
    # Predict
    scores = predict_with_context(text, context)
    
    # Get adaptive threshold
    threshold = get_user_threshold(conn)
    
    # Decide flag
    flagged = any(v >= threshold for v in scores.values())
    
    # Save
    message_id = save_message(conn, thread_id, text, scores)
    
    return message_id, scores, flagged, threshold

In [60]:
thread_id = "demo"

msg_id, scores, flagged, threshold = process_message(
    conn,
    thread_id,
    "you're dead to me",
    "[CTX0] haha bro you're amazing 😂"
)

print("Scores:", scores)
print("Threshold:", threshold)
print("Flagged:", flagged)
print("Message ID:", msg_id)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
Scores: {'toxic': 0.21400000154972076, 'severe_toxic': 0.0010000000474974513, 'obscene': 0.020999999716877937, 'threat': 0.003000000026077032, 'insult': 0.032999999821186066, 'identity_hate': 0.004999999888241291}
Threshold: 0.2
Flagged: True
Message ID: 3


In [61]:
# User says it's toxic
update_feedback(conn, msg_id, 1)

print("Feedback saved")

Feedback saved


In [62]:
msg_id, scores, flagged, threshold = process_message(
    conn,
    thread_id,
    "you're dead to me",
    "[CTX0] haha bro you're amazing 😂"
)

print("NEW Threshold:", threshold)
print("Flagged:", flagged)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
NEW Threshold: 0.2
Flagged: True


In [75]:
thread_id = "demo2"

messages = [
    ("you are useless", 1),
    ("nice work bro", 0),
    ("shut up", 1),
    ("that was actually impressive", 0),
    ("you're such an idiot", 1),
    ("lol you're crazy 😂", 0),
    ("I will destroy you", 1),
    ("good effort", 0),
    ("what a dumb move", 1),
    ("haha chill bro", 0),
    
    ("you're annoying", 1),
    ("great job man", 0),
    ("stop talking nonsense", 1),
    ("well played", 0),
    ("you make no sense", 1),
    ("respect bro", 0),
    ("this is garbage", 1),
    ("nice one", 0),
    ("you're pathetic", 1),
    ("solid work", 0),
]

In [76]:
def process_message(conn, thread_id, text, context):
    
    # Predict using given context
    scores = predict_with_context(text, context)
    
    # Get adaptive threshold
    threshold = get_user_threshold(conn)
    
    # Decide flag
    flagged = any(v >= threshold for v in scores.values())
    
    # Save message
    message_id = save_message(conn, thread_id, text, scores)
    
    return message_id, scores, flagged, threshold

In [79]:
msg_id, scores, flagged, threshold = process_message(
    conn,
    thread_id,
    "you're ridiculous",
    "[CTX0] casual chat between friends"
)

print("\n--- FINAL TEST ---")
print("Threshold:", threshold)
print("Flagged:", flagged)
print("Scores:", scores)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step

--- FINAL TEST ---
Threshold: 0.2
Flagged: True
Scores: {'toxic': 0.8220000267028809, 'severe_toxic': 0.05400000140070915, 'obscene': 0.3330000042915344, 'threat': 0.03500000014901161, 'insult': 0.35600000619888306, 'identity_hate': 0.0860000029206276}
